[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/herrrickshaw/mospi-dataset-analysis-public/blob/main/notebooks/mospi_live_reference.ipynb)

# MoSPI Dataset Analysis — Live Reference Notebook

Companion notebook to [`herrrickshaw/mospi-dataset-analysis`](https://github.com/herrrickshaw/mospi-dataset-analysis-public) — India's trade, currency, inflation and PLI-policy analysis built from MoSPI/RBI/TRADESTAT data.

**What this notebook does, in order:**
1. Pulls every dataset this repo has already collected, straight from GitHub (`main` branch) — always the latest committed version, no local files needed.
2. Adds two **genuinely live** cells that re-scrape RBI's own public archives at run time, so the currency/reserves numbers are current *right now*, not just as of this repo's last commit.
3. Gives you quick lookup + charting helpers so you can reference any figure in this analysis without re-reading the source bulletins.

**What this notebook can't do**: the MoSPI connector itself (`api.mospi.gov.in`) uses an undocumented auth/request format only exercised through the Claude Code MCP connector — it isn't reproducible as a plain API call from Colab. Everything below either reads this repo's own cached JSON (already pulled via that connector) or re-scrapes RBI's fully public, unauthenticated archives directly.

Run cells top to bottom. Runtime → no GPU needed.


## 1. Setup

In [ ]:
!pip -q install requests pandas matplotlib beautifulsoup4 lxml

import json
import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

REPO_RAW = "https://raw.githubusercontent.com/herrrickshaw/mospi-dataset-analysis-public/main"
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")


## 2. Load every dataset this repo has collected

Pulled live from GitHub each time you run this cell — always whatever is currently committed to `main`, no need to re-clone or re-download by hand.


In [ ]:
DATASETS = [
    "mospi_snapshot_2026-07-18",
    "cpi_statewise_trend_2025-01_to_2026-06",
    "wpi_national_trend_2025-01_to_2026-04",
    "rbi_forex_reserves_2015-01_to_2025-06",
    "rbi_usd_inr_exchange_rate_2015-01_to_2026-07",
    "rupee_real_exchange_rate_2015_to_2026",
    "tradestat_hsn_export_import_2018-19_to_2025-26",
    "hsn_historical_trends_2018-19_to_2025-26",
    "services_trade_and_overall_balance_2015-16_to_2024-25",
    "trade_data_cross_validation_2026-07-18",
    "fertiliser_fuel_price_transmission_2012_to_2026",
    "import_dependency_policy_gap_analysis_2026-07-18",
    "country_trade_deficit_and_policy_history_2026-07-18",
    "country_commodity_breakdown_2026-07-18",
    "sector_country_priority_and_pli_coverage_2026-07-18",
    "export_destination_priority_and_pli_coverage_2026-07-18",
    "gdp_growth_rate_2012-13_to_2025-26",
    "five_year_trade_currency_synthesis_2026-07-18",
]

data = {}
for name in DATASETS:
    url = f"{REPO_RAW}/data/{name}.json"
    r = requests.get(url, timeout=20)
    if r.status_code == 200:
        data[name] = r.json()
    else:
        print(f"MISSING ({r.status_code}): {name}")

print(f"Loaded {len(data)} / {len(DATASETS)} datasets.")


## 3. Quick reference — headline figures

In [ ]:
def summary(name):
    """Print the top-level keys and a short preview of a loaded dataset."""
    d = data[name]
    print(f"--- {name} ---")
    for k in d:
        v = d[k]
        preview = json.dumps(v)[:160]
        print(f"  {k}: {preview}")

summary("five_year_trade_currency_synthesis_2026-07-18")


In [ ]:
# FY2025-26 headline snapshot (matches reports/FY2025-26_sample_report.html in the repo)
trade = data["tradestat_hsn_export_import_2018-19_to_2025-26"]["totals"]
fy = "2025 - 2026"
exports_m = trade["export"][fy]
imports_m = trade["import"][fy]
deficit_m = exports_m - imports_m

print(f"FY2025-26 exports:  US$ {exports_m:>12,.1f} M")
print(f"FY2025-26 imports:  US$ {imports_m:>12,.1f} M")
print(f"FY2025-26 deficit:  US$ {deficit_m:>12,.1f} M")


## 4. Live re-scrape — RBI forex reserves (right now, not as of the last commit)

RBI's Weekly Statistical Supplement archive is fully enumerable (`rbi.org.in/scripts/WSSView.aspx?Id=N`, no key/login needed) — this repo's currency bulletin already documented the pattern. This cell walks backward from a recent guessed `Id` to find the latest published release, so the number below is live at the moment you run it.


In [ ]:
import re

def fetch_latest_rbi_reserves(start_id=28590, lookback=60):
    """Walk backward from start_id looking for the latest published WSS release
    with a parseable 'Total Reserves' row. Increase start_id over time as RBI
    publishes new weekly releases (~+1/week since 19 Sep 1998 = Id 2; Id 28579
    published 17 Jul 2026 was the latest confirmed release when this was written)."""
    row_pattern = re.compile(
        r"Total Reserves</td>\s*<td[^>]*>([\d,]+)</td>\s*<td[^>]*>([\d,]+)</td>"
    )
    date_pattern = re.compile(r"As on\s+([A-Za-z]+\.?\s+\d{1,2},\s+\d{4})")
    for wss_id in range(start_id, start_id - lookback, -1):
        url = f"https://rbi.org.in/scripts/WSSView.aspx?Id={wss_id}"
        try:
            r = requests.get(url, timeout=15)
        except requests.RequestException:
            continue
        if r.status_code != 200:
            continue
        m_row = row_pattern.search(r.text)
        m_date = date_pattern.search(r.text)
        if m_row and m_date:
            # column 1 = Rs Crore, column 2 = US$ Million (verified against RBI's own table)
            return {
                "wss_id": wss_id,
                "as_on": m_date.group(1),
                "total_reserves_rs_crore": float(m_row.group(1).replace(",", "")),
                "total_reserves_usd_million": float(m_row.group(2).replace(",", "")),
                "url": url,
            }
    return None

latest_reserves = fetch_latest_rbi_reserves()
if latest_reserves:
    print(f"Latest RBI reserves reading found live just now:")
    print(f"  As on:  {latest_reserves['as_on']}")
    print(f"  Total:  US$ {latest_reserves['total_reserves_usd_million']:,.1f} Million")
    print(f"  Source: {latest_reserves['url']}")
else:
    print("Could not locate a parseable release in this Id range — RBI may have changed"
          " its page markup, or start_id needs bumping upward (new IDs are added weekly).")


## 5. Live check — USD/INR rate (right now)

RBI's own Reference Rate Archive (`rbi.org.in/scripts/referenceratearchive.aspx`) is what this repo's currency bulletin scraped — but it's an ASP.NET WebForms postback (`__VIEWSTATE`/`__EVENTVALIDATION` hidden fields regenerated per request) that a plain GET with query params can't reach; RBI's WAF returns `418 Unauthorised Access` for that shortcut. Reproducing the full postback is possible but fragile and out of scope for a quick live-reference cell.

Instead, this cell uses the **Frankfurter API** (ECB reference rates, free, keyless, no scraping) as a live cross-check — it won't match RBI's official fixing to the last paisa (different rate-fixing methodology and time-of-day), but it's close enough to confirm the rupee's current level in real time. For the RBI figure itself, use this repo's own bulletin or extend the `session.post(...)` pattern in the raw HTML above (`txtFromDate`/`txtToDate`/`btnSubmit` fields, found by inspecting the form).


In [ ]:
def fetch_live_usd_inr():
    """Live USD/INR via Frankfurter (ECB reference rates) -- no key, no scraping fragility.
    Will differ slightly from RBI's own official reference rate (different fixing time/method)."""
    r = requests.get("https://api.frankfurter.app/latest", params={"from": "USD", "to": "INR"}, timeout=15)
    r.raise_for_status()
    payload = r.json()
    return payload["date"], payload["rates"]["INR"]

date, rate = fetch_live_usd_inr()
print(f"Live USD/INR (Frankfurter/ECB, {date}): Rs {rate}/US$")
print(f"Compare to this repo's own RBI-scraped figure: Rs 96.3665/US$ as of 2026-07-17")


## 6. On-demand TRADESTAT lookup (live)

Parameterized version of the scraper this repo used throughout — pass a 2-digit HS chapter and get every country's FY figures for that chapter, live, right now. Read-only, public form; no login required, following the same cookie + CSRF workaround documented in this repo's trade-balance bulletin.


In [ ]:
def fetch_tradestat_by_hs_code(hs_code, direction="export", currency="2"):
    """direction: 'export' or 'import'. currency: '2' = US$ Million, '1' = Rs Crore."""
    suffix = "cmace" if direction == "export" else "cmaci"
    base = "https://tradestat.commerce.gov.in"
    path = f"/eidb/commodity_wise_all_countries_{direction}"
    session = requests.Session()
    get_resp = session.get(base + path, timeout=20)
    csrf_match = re.search(r'name="_token" value="([^"]+)"', get_resp.text)
    if not csrf_match:
        return None, "Could not find CSRF token — TRADESTAT's form markup may have changed."
    token = csrf_match.group(1)
    payload = {
        "_token": token,
        f"Eidbhscode_{suffix}": hs_code,
        f"EidbYear_{suffix}": "2025",
        f"EidbReport_{suffix}": currency,
    }
    post_resp = session.post(base + path, data=payload, timeout=20)
    return post_resp.text, None

# Example: live-pull HS85 (electrical machinery) exports by country, right now
html, err = fetch_tradestat_by_hs_code("85", direction="export")
if err:
    print(err)
else:
    print(f"Fetched {len(html):,} characters of live TRADESTAT HTML for HS85 exports.")
    print("Parse with pandas.read_html(html) or BeautifulSoup for the country table.")


## 7. Five-year synthesis dashboard (from this repo's own capstone chart)

In [ ]:
synth = data["five_year_trade_currency_synthesis_2026-07-18"]
years = synth["trade_totals_usd_million"]["years"]
exp = synth["trade_totals_usd_million"]["export"]
imp = synth["trade_totals_usd_million"]["import"]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(years, exp, marker="o", label="Exports (US$ M)")
ax.plot(years, imp, marker="o", label="Imports (US$ M)")
ax.set_title("India merchandise trade, FY2021-22 to FY2025-26 (US$ Million)")
ax.set_ylabel("US$ Million")
ax.legend()
ax.grid(alpha=0.25)
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

print(f"Import growth: {synth['trade_totals_usd_million']['import_growth_5yr_pct']}%")
print(f"Export growth: {synth['trade_totals_usd_million']['export_growth_5yr_pct']}%")
print(f"Services cushion of extra goods gap: {synth['goods_services_balance_usd_billion']['services_cushion_pct_of_extra_goods_gap']}%")


## 8. FY2025-26 sample report reference table

Same figures as [`reports/FY2025-26_sample_report.html`](https://github.com/herrrickshaw/mospi-dataset-analysis-public/blob/main/reports/FY2025-26_sample_report.html) in the repo — all in US$ Million.


In [ ]:
hsn = data["hsn_historical_trends_2018-19_to_2025-26"]
top_imports = pd.DataFrame(
    [(r["hs"], r["name"], r["latest"], r["growth_pct"]) for r in hsn["top12_imports_by_value"]],
    columns=["HS", "Chapter", "FY2025-26 (US$ M)", "8yr Growth %"],
)
top_exports = pd.DataFrame(
    [(r["hs"], r["name"], r["latest"], r["growth_pct"]) for r in hsn["top12_exports_by_value"]],
    columns=["HS", "Chapter", "FY2025-26 (US$ M)", "8yr Growth %"],
)
print("Top 12 import chapters, FY2025-26 (US$ Million):")
display(top_imports)
print("\nTop 12 export chapters, FY2025-26 (US$ Million):")
display(top_exports)


## 9. A tour of Python visualization toolkits (beyond matplotlib)

Per [Mode's Python data-viz library roundup](https://mode.com/blog/python-data-visualization-libraries), the same trade data reads differently through different toolkits — useful to know which one fits a given job:
- **Seaborn** — statistical styling on top of matplotlib, good for quick, good-looking static charts
- **Plotly** — interactive, hoverable, zoomable web charts (this is the one to reach for if the chart needs to survive being clicked on)
- **Altair** — declarative grammar-of-graphics (Vega-Lite) — concise for faceted/layered charts
- **Folium** — Leaflet.js-based maps — a lighter-weight alternative to the repo's own hand-rolled amCharts choropleth
- **missingno** — visualizes data completeness/missing values — a quick sanity check before trusting a dataset


In [ ]:
!pip -q install seaborn plotly altair folium missingno


### 9a. Seaborn — top-12 import chapters, statistically styled

In [ ]:
import seaborn as sns

sns.set_theme(style="whitegrid")
top_imports_df = pd.DataFrame(hsn["top12_imports_by_value"])[["hs", "name", "latest", "growth_pct"]]
top_imports_df = top_imports_df.sort_values("latest", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
sns.barplot(data=top_imports_df, y="name", x="latest", hue="name", legend=False, palette="Blues_r", ax=ax)
ax.set_xlabel("FY2025-26 imports (US$ Million)")
ax.set_ylabel("")
ax.set_title("Top 12 import chapters — Seaborn styling")
plt.tight_layout()
plt.show()


### 9b. Plotly — interactive five-year trade trend (hover for exact values)

In [ ]:
import plotly.graph_objects as go

years = synth["trade_totals_usd_million"]["years"]
exp = synth["trade_totals_usd_million"]["export"]
imp = synth["trade_totals_usd_million"]["import"]

fig = go.Figure()
fig.add_trace(go.Scatter(x=years, y=exp, mode="lines+markers", name="Exports (US$ M)"))
fig.add_trace(go.Scatter(x=years, y=imp, mode="lines+markers", name="Imports (US$ M)"))
fig.update_layout(
    title="India merchandise trade, FY2021-22 to FY2025-26 — Plotly (hover for values)",
    yaxis_title="US$ Million",
    hovermode="x unified",
    template="plotly_white",
)
fig.show()


### 9c. Altair — declarative bar chart, top export destination countries

In [ ]:
import altair as alt

ed = data["export_destination_priority_and_pli_coverage_2026-07-18"]
export_countries_df = pd.DataFrame(
    [(r["country_display"], r["total_exposure_usd_m"], r["n_sectors"]) for r in ed["country_priority_ranking"][:10]],
    columns=["country", "total_exposure_usd_m", "n_sectors"],
)

chart = alt.Chart(export_countries_df).mark_bar().encode(
    x=alt.X("total_exposure_usd_m:Q", title="Combined exposure (US$ Million)"),
    y=alt.Y("country:N", sort="-x", title=None),
    color=alt.Color("n_sectors:Q", scale=alt.Scale(scheme="teals"), title="# growth sectors"),
    tooltip=["country", "total_exposure_usd_m", "n_sectors"],
).properties(title="Top export destinations — Altair", width=500, height=300)
chart


### 9d. Folium — map India's top trading partners (lighter-weight alternative to the repo's amCharts choropleth)

In [ ]:
import folium

# Approximate capital/centroid coordinates for quick mapping -- illustrative, not survey-grade.
COUNTRY_COORDS = {
    "China": (35.8617, 104.1954), "USA": (37.0902, -95.7129), "UAE": (23.4241, 53.8478),
    "Russia": (61.5240, 105.3188), "Saudi Arabia": (23.8859, 45.0792), "Netherlands": (52.1326, 5.2913),
    "Singapore": (1.3521, 103.8198), "United Kingdom": (55.3781, -3.4360), "Hong Kong": (22.3193, 114.1694),
    "Germany": (51.1657, 10.4515), "Iraq": (33.2232, 43.6793), "Switzerland": (46.8182, 8.2275),
}

sc = data["sector_country_priority_and_pli_coverage_2026-07-18"]
m = folium.Map(location=[20.5937, 78.9629], zoom_start=3, tiles="CartoDB positron")
folium.Marker([20.5937, 78.9629], popup="India", icon=folium.Icon(color="green")).add_to(m)

for r in sc["country_priority_ranking"][:10]:
    coords = COUNTRY_COORDS.get(r["country_display"])
    if not coords:
        continue
    folium.Circle(
        location=coords,
        radius=r["total_exposure_usd_m"] * 15,  # scaled for visibility, not to true distance units
        popup=f"{r['country_display']}: US$ {r['total_exposure_usd_m']:,.0f}M import exposure",
        color="#a5402c", fill=True, fill_opacity=0.4,
    ).add_to(m)

m


### 9e. missingno — sanity-check data completeness before trusting a dataset

In [ ]:
import missingno as msno

# Quick completeness check across the country-ranking records this notebook just loaded --
# a real habit worth having before citing any field from a newly-loaded dataset.
ranking_df = pd.DataFrame(sc["country_priority_ranking"])
print(f"Checking {len(ranking_df)} country-ranking rows for missing fields:")
msno.matrix(ranking_df)
plt.show()
print(ranking_df.isnull().sum())


---
**Source repo**: [herrrickshaw/mospi-dataset-analysis](https://github.com/herrrickshaw/mospi-dataset-analysis-public) — every JSON file loaded above lives under `data/`, every chart referenced lives under `charts/` and `reports/`. This notebook re-derives nothing new except the two live-scrape cells (§4–6); everything else mirrors data already collected there.
